<a href="https://colab.research.google.com/github/tougheye/Data_processing/blob/main/column_parsing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import pandas as pd

file_path = "/content/One_Change_prep.xlsx"
initial_df = pd.read_excel(file_path)

initial_df.head()

,PhysicianUnionizationSurvey_Id,RAW_Most_Significant_Factor
0,915,Compensation
1,644,"Compensation, Work/life balance, Staffing"
2,854,"Compensation, ""Leaves (parental, medical, sick..."
3,1046,"Compensation, ""Leaves (parental, medical, sick..."
4,563,"Compensation, Retirement, Work/life balance"


split column that has internal comma but needs to be parsed out based on comma

In [6]:
import csv
import pandas as pd

def parse_csv_column(val):
  if pd.isna(val):
    return val
  # csv.reader expects an iterable of strings, so we pass a list containing the single string val
  return next(csv.reader([val], skipinitialspace=True))

initial_df['response_list'] = initial_df['RAW_Most_Significant_Factor'].apply(parse_csv_column)
initial_df.head()

,PhysicianUnionizationSurvey_Id,RAW_Most_Significant_Factor,response_list
0,915,Compensation,[Compensation]
1,644,"Compensation, Work/life balance, Staffing","[Compensation, Work/life balance, Staffing]"
2,854,"Compensation, ""Leaves (parental, medical, sick...","[Compensation, Leaves (parental, medical, sick..."
3,1046,"Compensation, ""Leaves (parental, medical, sick...","[Compensation, Leaves (parental, medical, sick..."
4,563,"Compensation, Retirement, Work/life balance","[Compensation, Retirement, Work/life balance]"


In [8]:
initial_df['options_list'] = initial_df['RAW_Most_Significant_Factor'].str.split(r',\s*(?![^()]*\))', regex=True)
initial_df.head()

,PhysicianUnionizationSurvey_Id,RAW_Most_Significant_Factor,options_list
0,915,Compensation,[Compensation]
1,644,"Compensation, Work/life balance, Staffing","[Compensation, Work/life balance, Staffing]"
2,854,"Compensation, ""Leaves (parental, medical, sick...","[Compensation, ""Leaves (parental, medical, sic..."
3,1046,"Compensation, ""Leaves (parental, medical, sick...","[Compensation, ""Leaves (parental, medical, sic..."
4,563,"Compensation, Retirement, Work/life balance","[Compensation, Retirement, Work/life balance]"


In [14]:
initial_df['options_list'].str.len().max()

9.0

In [21]:
# get the number of columns needed
col_n = int(initial_df['options_list'].str.len().max())

column_names = [f"Option_{i+1}" for i in range(col_n)]

clean = initial_df['options_list'].apply(lambda x: x if isinstance(x, list) else [])

expanded_df = pd.DataFrame(clean.tolist(), columns=column_names, index=initial_df.index)
expanded_df.head()

,Option_1,Option_2,Option_3,Option_4,Option_5,Option_6,Option_7,Option_8,Option_9
0,Compensation,None,None,None,None,None,None,None,None
1,Compensation,Work/life balance,Staffing,None,None,None,None,None,None
2,Compensation,"""Leaves (parental, medical, sick, etc.)""",Protected time,Retirement,Work/life balance,None,None,None,None
3,Compensation,"""Leaves (parental, medical, sick, etc.)""",Protected time,Work/life balance,Staffing,None,None,None,None
4,Compensation,Retirement,Work/life balance,None,None,None,None,None,None


In [23]:
final_df = pd.concat([initial_df, expanded_df], axis=1)
final_df = final_df.fillna("")
final_df

,PhysicianUnionizationSurvey_Id,RAW_Most_Significant_Factor,options_list,Option_1,Option_2,Option_3,Option_4,Option_5,Option_6,Option_7,Option_8,Option_9
0,915,Compensation,[Compensation],Compensation,,,,,,,,
1,644,"Compensation, Work/life balance, Staffing","[Compensation, Work/life balance, Staffing]",Compensation,Work/life balance,Staffing,,,,,,
2,854,"Compensation, ""Leaves (parental, medical, sick...","[Compensation, ""Leaves (parental, medical, sic...",Compensation,"""Leaves (parental, medical, sick, etc.)""",Protected time,Retirement,Work/life balance,,,,
3,1046,"Compensation, ""Leaves (parental, medical, sick...","[Compensation, ""Leaves (parental, medical, sic...",Compensation,"""Leaves (parental, medical, sick, etc.)""",Protected time,Work/life balance,Staffing,,,,
4,563,"Compensation, Retirement, Work/life balance","[Compensation, Retirement, Work/life balance]",Compensation,Retirement,Work/life balance,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...
1074,1088,"Compensation, Protected time, Work/life balanc...","[Compensation, Protected time, Work/life balan...",Compensation,Protected time,Work/life balance,Staffing,,,,,
1075,1087,"Compensation, Protected time, Retirement, Work...","[Compensation, Protected time, Retirement, Wor...",Compensation,Protected time,Retirement,Work/life balance,Staffing,,,,
1076,1086,"Compensation, Limits on outside income","[Compensation, Limits on outside income]",Compensation,Limits on outside income,,,,,,,
1077,1085,"Compensation, ""Leaves (parental, medical, sick...","[Compensation, ""Leaves (parental, medical, sic...",Compensation,"""Leaves (parental, medical, sick, etc.)""",Retirement,Work/life balance,Staffing,,,,


In [24]:
final_df.to_excel("processed_one_issue.xlsx", index=False)
#